# Football Player Positioning AI - LSTM Training

Train a per-player LSTM model on Google Colab with free GPU.

**Important**: Each game's player is a DIFFERENT person.

## 0. Configuration

All configurable parameters in one place.

In [ ]:
# ============================================================
#  ALL CONFIGURABLE PARAMETERS
# ============================================================

# --- Target ---
GAME = 'game1'           # game1, game2, game3
PLAYER = 'home_14'       # e.g. home_14, away_9, home_6

# --- Window (seconds) ---
OBS_SECONDS = 3          # Observation window: how many seconds of history
PRED_SECONDS = 2         # Prediction window: how many seconds to predict
SAMPLE_RATE = 25         # FPS of tracking data
STRIDE_FRAMES = 5        # Sliding window step (frames)

# --- Training ---
EPOCHS = 300             # Max epochs (early stopping will cut short)
BATCH_SIZE = 32          # Batch size
LR = 0.0001              # Learning rate

# --- Derived ---
DATASET_ID = f'{GAME}_{PLAYER}'
OBS_FRAMES = OBS_SECONDS * SAMPLE_RATE
PRED_FRAMES = PRED_SECONDS * SAMPLE_RATE

# ============================================================
import os
os.environ['OBS_SECONDS'] = str(OBS_SECONDS)
os.environ['PRED_SECONDS'] = str(PRED_SECONDS)
os.environ['SAMPLE_RATE'] = str(SAMPLE_RATE)
os.environ['STRIDE_FRAMES'] = str(STRIDE_FRAMES)
os.environ['EPOCHS'] = str(EPOCHS)
os.environ['BATCH_SIZE'] = str(BATCH_SIZE)
os.environ['LR'] = str(LR)
os.environ['PRED_FRAMES'] = str(PRED_FRAMES)

print(f'[CONFIG] Target:      {DATASET_ID}')
print(f'[CONFIG] Observe:     {OBS_SECONDS}s = {OBS_FRAMES} frames')
print(f'[CONFIG] Predict:     {PRED_SECONDS}s = {PRED_FRAMES} frames')
print(f'[CONFIG] Training:    epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}')

## 1. Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)')

## 2. Clone Repository

In [ ]:
import os
if os.path.exists('football-positioning-ai'):
    !cd football-positioning-ai && git pull
else:
    !git clone https://github.com/Sysus611/football-positioning-ai.git
%cd football-positioning-ai
!pip install pandas numpy pyarrow scipy pyyaml matplotlib -q
print('[OK] Ready')

## 3. Download Data

In [ ]:
import os, urllib.request
BASE_URL = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data"
FILES = {
    'game1': {
        'tracking_home.csv': 'Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv',
        'tracking_away.csv': 'Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv',
    },
    'game2': {
        'tracking_home.csv': 'Sample_Game_2/Sample_Game_2_RawTrackingData_Home_Team.csv',
        'tracking_away.csv': 'Sample_Game_2/Sample_Game_2_RawTrackingData_Away_Team.csv',
    },
    'game3': {
        'tracking.txt': 'Sample_Game_3/Sample_Game_3_tracking.txt',
        'metadata.xml': 'Sample_Game_3/Sample_Game_3_metadata.xml',
        'metadata.json': 'Sample_Game_3/Sample_Game_3_events.json',
    },
}
file_map = FILES[GAME]
game_dir = f'data/raw/metrica/{GAME}'
os.makedirs(game_dir, exist_ok=True)
for local_name, remote_path in file_map.items():
    dest = f'{game_dir}/{local_name}'
    if os.path.exists(dest):
        print(f'  [SKIP] {local_name}')
        continue
    print(f'  [DOWNLOAD] {remote_path}...', end='')
    urllib.request.urlretrieve(f'{BASE_URL}/{remote_path}', dest)
    print(f' OK ({os.path.getsize(dest)/1024/1024:.1f} MB)')
print('[OK] Done')

## 4. Preprocessing

In [ ]:
import time; t0 = time.time()
!python src/preprocessing/preprocess.py
print(f'[OK] {time.time()-t0:.0f}s')

## 5. Feature Engineering

In [ ]:
import time; t0 = time.time()
!OBS_SECONDS={OBS_SECONDS} PRED_SECONDS={PRED_SECONDS} SAMPLE_RATE={SAMPLE_RATE} STRIDE_FRAMES={STRIDE_FRAMES} \
  python src/features/build_features.py {GAME} {PLAYER}
print(f'[OK] {time.time()-t0:.0f}s')

## 6. Train

In [ ]:
import time; t0 = time.time()
!EPOCHS={EPOCHS} BATCH_SIZE={BATCH_SIZE} LR={LR} PRED_FRAMES={PRED_FRAMES} \
  python src/training/train.py {DATASET_ID}
print(f'\n[OK] {time.time()-t0:.0f}s ({(time.time()-t0)/60:.1f} min)')

## 7. Loss Curve

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np

with open(f'data/models/{DATASET_ID}_history.json') as f:
    history = json.load(f)

train_l = history['train_losses']
val_l = history['val_losses']
best_ep = history['best_epoch']
epochs_ran = len(train_l)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f'{DATASET_ID} Training History', fontsize=14, fontweight='bold')

# --- Loss Curve ---
x = range(1, epochs_ran + 1)
ax1.plot(x, train_l, 'b-', alpha=0.7, label='Train Loss')
ax1.plot(x, val_l, 'r-', alpha=0.7, label='Val Loss')
ax1.axvline(x=best_ep, color='green', ls='--', alpha=0.5, label=f'Best epoch ({best_ep})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (weighted MSE)')
ax1.set_title('Loss Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# --- Distance Error Curve ---
PITCH_AVG = (105 + 68) / 2
train_dist = [np.sqrt(l) * PITCH_AVG for l in train_l]
val_dist = [np.sqrt(l) * PITCH_AVG for l in val_l]
ax2.plot(x, train_dist, 'b-', alpha=0.7, label='Train')
ax2.plot(x, val_dist, 'r-', alpha=0.7, label='Validation')
ax2.axvline(x=best_ep, color='green', ls='--', alpha=0.5, label=f'Best ({best_ep})')
best_dist = np.sqrt(history['best_val_loss']) * PITCH_AVG
ax2.axhline(y=best_dist, color='red', ls=':', alpha=0.3)
ax2.annotate(f'{best_dist:.2f}m', xy=(epochs_ran, best_dist), fontsize=11, color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Distance Error (m)')
ax2.set_title('~Average Prediction Error')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best: epoch {best_ep}, val_loss={history["best_val_loss"]:.6f} (~{best_dist:.2f}m)')

## 8. Visualization: Prediction vs Actual

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt, sys
from matplotlib.patches import Arc, Circle
sys.path.insert(0, '.')
from src.model.lstm_baseline import PlayerLSTM

def draw_pitch(ax, L=105, W=68):
    ax.set_facecolor('#1a472a')
    ax.set_xlim(-3, L+3); ax.set_ylim(-3, W+3)
    ax.set_aspect('equal'); ax.tick_params(colors='#aaa', labelsize=7)
    lw, c, hw = 1.2, 'white', W/2
    for pts in [([0,0],[0,W]),([0,L],[W,W]),([L,L],[W,0]),([L,0],[0,0])]:
        ax.plot(*pts, c, lw=lw)
    ax.plot([L/2,L/2],[0,W], c, lw=lw)
    ax.add_patch(Circle((L/2,hw), 9.15, fill=False, ec=c, lw=lw))
    for s in [0, L]:
        d = 1 if s==0 else -1
        ax.plot([s,s+d*16.5],[hw+20.16,hw+20.16], c, lw=lw)
        ax.plot([s+d*16.5,s+d*16.5],[hw+20.16,hw-20.16], c, lw=lw)
        ax.plot([s+d*16.5,s],[hw-20.16,hw-20.16], c, lw=lw)
        ax.plot([s,s+d*5.5],[hw+9.16,hw+9.16], c, lw=lw)
        ax.plot([s+d*5.5,s+d*5.5],[hw+9.16,hw-9.16], c, lw=lw)
        ax.plot([s+d*5.5,s],[hw-9.16,hw-9.16], c, lw=lw)

ckpt = torch.load(f'data/models/{DATASET_ID}.pt', map_location='cpu', weights_only=False)
model = PlayerLSTM(input_dim=ckpt['input_dim'], pred_frames=ckpt['pred_frames'])
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

data = np.load(f'data/tensors/{DATASET_ID}.npz')
X_val, Y_val = data['X_val'], data['Y_val']
L, W = 105, 68

fig, axes = plt.subplots(3, 3, figsize=(24, 16))
fig.suptitle(f'{DATASET_ID} — Pred vs Actual ({OBS_SECONDS}s obs → {PRED_SECONDS}s pred)', fontsize=18, color='white', fontweight='bold')
fig.patch.set_facecolor('#111')

for ax in axes.flat:
    idx = np.random.randint(len(X_val))
    x_sample = X_val[idx]
    last = x_sample[-1]
    with torch.no_grad():
        pred = model(torch.from_numpy(X_val[idx:idx+1])).numpy()[0]
    actual = Y_val[idx]
    draw_pitch(ax)
    # Other players
    others = last[0:42].reshape(21, 2)
    for i in range(21):
        px, py = others[i,0]*L, others[i,1]*W
        if px==0 and py==0: continue
        ax.scatter(px, py, c='#4da6ff' if i<10 else '#ff6b6b', s=60, edgecolors='white', linewidths=0.5, zorder=6, alpha=0.8)
    # Ball
    ax.scatter(last[42]*L, last[43]*W, c='yellow', s=100, edgecolors='black', linewidths=1, zorder=8, label='Ball')
    # Past trajectory
    ox, oy = x_sample[:,44]*L, x_sample[:,45]*W
    ax.plot(ox, oy, 'c-', lw=2, alpha=0.5, label=f'Past {OBS_SECONDS}s')
    ax.scatter(ox[-1], oy[-1], c='cyan', s=80, edgecolors='white', linewidths=1.5, zorder=9)
    # Actual
    ax.plot(actual[:,0]*L, actual[:,1]*W, 'w-', lw=2.5, alpha=0.9, label='Actual')
    ax.scatter(actual[-1,0]*L, actual[-1,1]*W, c='white', s=100, marker='*', zorder=9)
    # Predicted
    ax.plot(pred[:,0]*L, pred[:,1]*W, '#ff4444', ls='--', lw=2.5, alpha=0.9, label='Predicted')
    ax.scatter(pred[-1,0]*L, pred[-1,1]*W, c='#ff4444', s=100, marker='*', zorder=9)
    err = np.sqrt(((pred-actual)**2).sum(1)).mean()*((L+W)/2)
    ax.set_title(f'#{idx} avg:{err:.1f}m', color='white', fontsize=11)
    ax.legend(fontsize=7, loc='upper right', framealpha=0.6)

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig('prediction_vis.png', dpi=150, bbox_inches='tight', facecolor='#111')
plt.show()

## 9. Download Model

In [ ]:
from google.colab import files
files.download(f'data/models/{DATASET_ID}.pt')